In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
import joblib

In [8]:
df = pd.read_csv("datasets/final_ds.csv")
target_cols = [f"NCAR{i}" for i in range(1,21)]
subset = target_cols + ['stage', 'date']
df_clean = df.dropna(subset=subset).reset_index(drop=True)

In [9]:
df_clean['date'] = pd.to_datetime(df_clean['date'])

In [10]:
train_mask = (df_clean['date'] >= '2025-01-01') & (df_clean['date'] <= '2025-11-30')
test_mask = (df_clean['date'] >= '2026-01-01') & (df_clean['date'] <= '2026-06-30')

In [12]:
models = {}
eval_metrics = []

for dayX in range(1,21):
    target = f'NCAR{dayX}'
    features = ['stage', 
                'polarity',
                'drug_count',
                'Total Revenue', 
                'Operating Revenue', 
                'Cash And Cash Equivalents', 
                'Capital Expenditure', 
                'Stock_Trend_30d', 
                'Stock_Trend_60d',
                'NBI_Trend_30d', 
                'NBI_Trend_60d',
                'isDelisted']


    X_train = df_clean.loc[train_mask, features]
    y_train = df_clean.loc[train_mask, target]

    X_test = df_clean.loc[test_mask, features]
    y_test = df_clean.loc[test_mask, target]

    X_train['stage'] = X_train['stage'].astype('category')
    X_test['stage'] = X_test['stage'].astype('category')

    X_train['isDelisted'] = X_train['isDelisted'].astype('int')
    X_test['isDelisted'] = X_test['isDelisted'].astype('int')

    cont_vars = [c for c in features if (c != 'stage' and c!= 'isDelisted')]
    X_train[cont_vars] = X_train[cont_vars].apply(pd.to_numeric)
    X_test[cont_vars] = X_test[cont_vars].apply(pd.to_numeric)

    y_train = y_train.apply(pd.to_numeric)
    y_test = y_test.apply(pd.to_numeric)

    print(f"Train observation count: {len(X_train)}")
    print(f"Test observation count: {len(X_test)}")

    model = xgb.XGBRegressor(
        objective = 'reg:pseudohubererror',
        huber_slope = 1.0,
        eval_metric = 'mae',
        n_estimators = 100,
        learning_rate = 0.05,
        max_depth=5,
        random_state = 42,
        n_jobs = 1,
        tree_method = "hist",
        enable_categorical = True,
        missing = np.nan
    )


    model.fit(X_train,
            y_train,
            eval_set = [(X_test, y_test)],
            verbose= False
            )

    models[dayX] = model

    y_pred = model.predict(X_test)

    # model evaluation

    rank_ic, p_val = spearmanr(y_test, y_pred)

    eval_metrics.append({
        "Day": dayX,
        "Directional Accuracy": np.mean(np.sign(y_pred) == np.sign(y_test)),
        "Rank IC": rank_ic, 
        "p_val": p_val,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

    importance = pd.DataFrame({
        'Features' : features,
        'Importance': models[dayX].feature_importances_
    }).sort_values('Importance', ascending = False)

    print(importance)

    labels = range(1,11)

    deciles = pd.qcut(y_pred, q=10, labels = labels)
    spread = y_test.groupby(deciles).mean()

    print(spread)

Train observation count: 2316
Test observation count: 1477
                     Features  Importance
5   Cash And Cash Equivalents    0.157563
7             Stock_Trend_30d    0.096312
6         Capital Expenditure    0.093835
8             Stock_Trend_60d    0.093252
4           Operating Revenue    0.091323
3               Total Revenue    0.089463
9               NBI_Trend_30d    0.084372
0                       stage    0.080550
10              NBI_Trend_60d    0.079573
1                    polarity    0.073097
2                  drug_count    0.060661
11                 isDelisted    0.000000
1    -0.028480
2    -0.028918
3     0.055987
4     0.070865
5    -0.001229
6    -0.008553
7    -0.008726
8     0.030228
9     0.002155
10   -0.009361
Name: NCAR1, dtype: float64
Train observation count: 2316
Test observation count: 1477
                     Features  Importance
5   Cash And Cash Equivalents    0.216183
3               Total Revenue    0.144630
1                    polarity   

In [13]:
eval_metrics = pd.DataFrame(eval_metrics)
print(eval_metrics)

    Day  Directional Accuracy   Rank IC         p_val      RMSE       MAE  \
0     1              0.503724  0.108589  2.888179e-05  0.358596  0.075713   
1     2              0.487475  0.104155  6.061836e-05  1.043889  0.114501   
2     3              0.514557  0.120397  3.483467e-06  0.806510  0.134648   
3     4              0.499661  0.097291  1.805364e-04  0.745149  0.146896   
4     5              0.505755  0.114648  1.000888e-05  0.747738  0.162339   
5     6              0.544347  0.189769  1.925128e-13  0.743110  0.170268   
6     7              0.528097  0.176066  9.486879e-12  0.556332  0.169906   
7     8              0.532837  0.186789  4.609871e-13  0.720398  0.192266   
8     9              0.506432  0.160282  5.835388e-10  1.041748  0.215988   
9    10              0.499661  0.152382  3.962533e-09  0.977733  0.213742   
10   11              0.513202  0.179814  3.365697e-12  1.012205  0.219658   
11   12              0.520650  0.164399  2.069752e-10  0.584071  0.208411   

In [14]:
joblib.dump(models, "xgb_models_1_to_20.joblib")

['xgb_models_1_to_20.joblib']